---
title: "DRG Checks"

author: "Carlos Resurreccion"

date: "2025-04-03"
---


# Parameters

Change which year_to_load to process in
`~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in
`~/pids-drg-claims/data-cleaning/00a-parameters.r`

Change seldom touched parameters in
`~/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`


In [1]:
source(here::here("data-cleaning", "00a-parameters.r"))


Parallelization: TRUE 


# Libraries


In [2]:
source(here::here("data-cleaning", "00b-packages.r"))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc/pids-drg-claims

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy


Loading required package: future

Loading required package: future.apply

Load

# R Scripts


In [3]:
source(here::here("data-cleaning", "00c-load-params-and-scripts.r"))


==== Loaded Parameters ====

year_to_load: 2018

Automate: FALSE


Sourcing scripts from:/home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R

✅ Authentication successful using service account key.

All directories exist.


Total Rows via cached object: 11777674

Utilizing 4 cores (8 threads)


Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.2.0.process_helper_functions.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.3.0.grouping_functions.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/1.0.query_bq_to_dt.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/2.0.split_and_save_part.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/3.0.create_sample_files.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v

# Load Mapping Data


In [4]:
source(here::here("data-cleaning", "00d-load-mapping.r"))


==== Loading Data Mapping ====

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/proc.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/rvs_icd9.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/acr_rvs.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/tdrg_icd10.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/phl_icd10.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/i10vx.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/hci_2018.rds

00d-load-mapping.r successfully executed.



# Data Verification Proper


## Load final .rds


In [5]:
dt <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_b_bq_subset", ".rds"
  )
))


## Function Definitions


In [6]:
test_checks <- function(section_id) {
  # Coerce to two-character string (e.g., 1 → "01")
  section_id <- sprintf("%02d", as.integer(section_id))

  # Get the calling environment (e.g., global or wherever this is invoked from)
  calling_env <- parent.frame()

  # Build pattern to match only variables for the given section
  pattern <- paste0("^chk_", section_id, "_\\d{2}_.+")

  # List relevant check variables in the calling environment
  chk_vars <- ls(envir = calling_env, pattern = pattern)

  # If no matching checks found, warn and exit
  if (length(chk_vars) == 0) {
    message("⚠️ No checks found for section: ", section_id)
    return(invisible(NULL))
  }

  # Get values (assumed format: c(flag, info))
  chk_values_raw <- lapply(chk_vars, get, envir = calling_env)

  # Extract just the logical flag from each
  chk_flags <- sapply(chk_values_raw, function(x) isTRUE(x[1]))

  # Check for failures
  if (any(!chk_flags)) {
    failed_checks <- chk_vars[!chk_flags]

    # Build detailed failure messages
    failure_messages <- mapply(function(var, val) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", var)
      info <- if (length(val) > 1) val[2] else "No additional info"
      paste0("• ", suffix, ": ", info)
    }, var = failed_checks, val = chk_values_raw[!chk_flags], SIMPLIFY = TRUE)

    stop(paste0(
      "❌ Validation failed in section ", section_id, ":\n",
      paste(failure_messages, collapse = "\n")
    ))
  } else {
    # Print all check results
    cat(paste0("✅ All validation checks passed for section ", section_id, ":\n"))
    for (i in seq_along(chk_vars)) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", chk_vars[i])
      cat(paste0(suffix, ": ", chk_values_raw[[i]][1], "\n"))
    }
  }
}

debug_setequal <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    # Identify elements missing and extra
    missing_in_y <- setdiff(x, y) # present in x but missing in y
    extra_in_y <- setdiff(y, x) # present in y but not in x

    # Compose detailed message
    mismatch_msg <- paste0(
      if (length(missing_in_y)) {
        paste0("\n  - missing: ", paste(missing_in_y, collapse = ", "))
      } else {
        ""
      },
      if (length(extra_in_y)) {
        paste0("\n  - invalid: ", paste(extra_in_y, collapse = ", "))
      } else {
        ""
      }
    )

    return(c(FALSE, mismatch_msg))
  }
}


## Test Batch 01:


In [7]:
cols_expected <- bq_cols
cols_actual <- colnames(dt)
cols_schema <- fromJSON(here(
  "data-cleaning/r_scripts_v2",
  "bq_schema_cleaning.json"
))$name
chk_01_01_cols_match_expected <- debug_setequal(cols_expected, cols_actual)
chk_01_02_cols_match_schema <- debug_setequal(cols_schema, cols_actual)
test_checks(1)


✅ All validation checks passed for section 01:
cols_match_expected: TRUE
cols_match_schema: TRUE


## Test Batch 02:


In [8]:
nrow_expected <- fread(file = here(
  raw_claims_path,
  paste0(full_claims_prefix, year_to_load, file_type)
), select = 1L)[, .N]
nrow_actual_full <- nrow(dt)
chk_02_01_nrows_match_expected_full <- debug_setequal(
  nrow_expected, nrow_actual_full
)
nrow_actual_partial <- nrow_partial <- 0
for (loop_part in 1:split_parts) {
  nrow_partial <- nrow(read_appropriate_file(loop_part))
  nrow_actual_partial <- nrow_actual_partial + nrow_partial
}
chk_02_02_nrows_match_expected_partial <- debug_setequal(
  nrow_expected, nrow_actual_partial
)
test_checks(2)


✅ All validation checks passed for section 02:
nrows_match_expected_full: TRUE
nrows_match_expected_partial: TRUE
